#### Application Codebase Knowledge Graph (KG) Tools

##### Environment setup

###### Package imports

In [50]:
import os
from dotenv import load_dotenv
import time
import requests
import pandas as pd
from pathlib import Path
import importlib
from neo4j import GraphDatabase

###### Test connections

In [9]:
def test_neo4j_connection(env_file=Path.cwd().parent / ".env"):
    """
    Test connectivity to a Neo4j database using settings
    stored in a .env file.

    Expected variables:

        NEO4J_URI
        NEO4J_USERNAME
        NEO4J_PASSWORD
        NEO4J_DATABASE
    """

    # Load environment variables
    load_dotenv(env_file, override=True)

    uri = os.getenv("NEO4J_URI")
    username = os.getenv("NEO4J_USERNAME")
    password = os.getenv("NEO4J_PASSWORD")
    database = os.getenv("NEO4J_DATABASE")

    try:
        with GraphDatabase.driver(uri, auth=(username, password), database=database) as driver:

            # Test the connection
            driver.verify_connectivity()

            # Open a session
            with driver.session(database=database) as session:
                result = session.run(
                    """
                    RETURN
                        1 AS connected,
                        datetime() AS server_time
                    """
                )

                record = result.single()

                print("✅ Successfully connected to Neo4j")
                print(f"Database    : {database}")
                print(f"URI         : {uri}")
                print(f"Server Time : {record['server_time']}")

                return True

    except Exception as ex:
        print("❌ Connection failed")
        print(type(ex).__name__)
        print(ex)

        return False


test_neo4j_connection()

✅ Successfully connected to Neo4j
Database    : appkg
URI         : neo4j://127.0.0.1:7687
Server Time : 2026-09-09T01:05:57.593000000+00:00


True

In [12]:
def test_github_connection():
    """Test connectivity to GitHub and validate the GitHub PAT."""

    # Find .env in the directory above the notebook's working directory
    env_file = Path.cwd().parent / ".env"

    print(f"Loading environment from: {env_file}")

    if not env_file.exists():
        print("❌ .env file not found")
        return False

    load_dotenv(env_file, override=True)

    token = os.getenv("GITHUB_TOKEN")

    if not token:
        print("❌ GITHUB_TOKEN is missing from .env")
        return False

    # GitHub API headers
    headers = {
        "Accept": "application/vnd.github+json",
        "Authorization": f"Bearer {token}",
        "X-GitHub-Api-Version": "2026-03-10",
    }

    try:
        # Test GitHub API + PAT authentication
        response = requests.get(
            "https://api.github.com/user",
            headers=headers,
            timeout=10,
        )

        print(f"GitHub API status: {response.status_code}")

        if response.status_code == 200:
            user = response.json()

            print("✅ Successfully connected to GitHub")
            print("✅ PAT is valid")
            print(f"GitHub user : {user.get('login')}")
            print(f"User ID     : {user.get('id')}")
            print(f"Account type: {user.get('type')}")

            return True

        elif response.status_code == 401:
            print("❌ GitHub API connection succeeded")
            print("❌ PAT authentication failed")
            print("The token may be invalid, expired, or revoked.")

        elif response.status_code == 403:
            print("⚠️ GitHub API responded with 403 Forbidden")
            print("The PAT was received, but GitHub denied the request.")
            print(response.text)

        else:
            print("❌ GitHub API request failed")
            print(response.text)

        return False

    except requests.exceptions.Timeout:
        print("❌ Connection to GitHub timed out")
        return False

    except requests.exceptions.ConnectionError as ex:
        print("❌ Could not connect to GitHub")
        print(ex)
        return False

    except requests.exceptions.RequestException as ex:
        print("❌ GitHub request failed")
        print(type(ex).__name__)
        print(ex)
        return False

test_github_connection()

Loading environment from: E:\Projects\knowledge_graph_tools\.env
GitHub API status: 200
✅ Successfully connected to GitHub
✅ PAT is valid
GitHub user : georgejaymcmc
User ID     : 88515517
Account type: User


True

##### Repo details

In [70]:
# Keep your mapping as a class attribute (cleaner)
FILE_TYPES = {
    ".py": "Python Script",
    ".md": "Markdown",
    ".txt": "Text",
    ".json": "JSON",
    ".csv": "CSV",
    ".html": "HTML",
    ".js": "Javascript",
    ".css": "CSS",
    ".java": "Java",
    ".cpp": "C++",
    ".c": "C",
    ".h": "Header",
    ".sh": "Shell",
    ".jpg": "JPG",
    ".jpeg": "JPEG",
    ".gif": "GIF",
    ".svg": "SVG",
    ".ico": "ICO",
    ".pyi": "Python_Image",
    ".pyw": "Python_Web",
    ".pyx": "Python_Xml",
    ".bmp": "BMP",
    ".ts": "Typescript",
    ".ipynb": "Jupyter",
    ".ftl": "FreeMarker_Java_template",
    ".dtd": "DTD_XML_def",
    ".properties": "Java_prop",
    ".xhtml": "XHTML",
    ".xml": "XML",
    ".jsx": "Javascript",
    ".yml": "YAML",
    ".gitignore": ".gitignore",
    ".gitmodules": ".gitmodules",
    ".ini": ".ini",
    ".manifest": ".manifest",
    ".jsm": ".jsm",
    ".rdf": "RDF",
    ".zip": "ZIP",
    ".pdf": "PDF",
    ".sqlite": "SQLite",
    ".png": "PNG",
    ".epub": "Epub",
    ".opf": "OPF",
    ".lua": "LUA",
    ".opml": "OPML",
    ".rss": "RSS",
    ".atom": "ATOM",
    ".xpi": "XPI",
    ".csl": "CSL",
    ".scss": "CSS",
    ".woff": "WOFF",
    ".sql": "SQL",
    ".vbs": "VBScript",
    ".idl": "IDL",
    ".mjs": "Javascript",
    ".xul": "XUL",
    ".nsi": "NSI",
    ".nsh": "NSH",
    ".rc": "RC",
    ".nlf": "NLF"
}

DEFAULT_FILE_TYPE = "Other"

# src/codebase_kg/services/github_repo_service.py
class GithubRepoService:
    """
    Service to fetch GitHub repository metadata.
    All public methods are automatically exposed as CLI commands.
    """

    def __init__(self, token=None):
        self.headers = {}
        if token:
            self.headers["Authorization"] = f"Bearer {token}"
        # cache to avoid multiple API calls
        self._repo_cache = {}

    def _repo_data(self, owner, repo):
        """Get repository metadata, using the cache when available."""

        cache_key = f"{owner}/{repo}"

        if cache_key not in self._repo_cache:
            url = f"https://api.github.com/repos/{owner}/{repo}"

            response = requests.get(
                url,
                headers=self.headers,
                timeout=10
            )

            if response.status_code != 200:
                raise RuntimeError(
                    f"GitHub API error {response.status_code}: "
                    f"{response.text}"
                )

            self._repo_cache[cache_key] = response.json()

        return self._repo_cache[cache_key]

    def description(self, owner, repo):
        return self._repo_data(owner, repo).get("description")

    def language(self, owner, repo):
        return self._repo_data(owner, repo).get("language")

    def size(self, owner, repo):
        return self._repo_data(owner, repo).get("size")

    def visibility(self, owner, repo):
        return self._repo_data(owner, repo).get("visibility")

    def default_branch(self, owner, repo):
        return self._repo_data(owner, repo).get("default_branch")

    def stars(self, owner, repo):
        return self._repo_data(owner, repo).get("stargazers_count")

    # -----------------------------
    # Directory / File counts
    # -----------------------------
    def get_directory_file_count(self, owner, repo):
        """
        Returns a tuple: (directory_count, file_count)
        """
        # Get repo metadata to find default branch
        repo_data = self._repo_data(owner, repo)
        branch = repo_data.get("default_branch", "main")

        # Fetch the full repo tree recursively
        url = f"https://api.github.com/repos/{owner}/{repo}/git/trees/{branch}?recursive=1"
        response = requests.get(url, headers=self.headers)
        if response.status_code != 200:
            raise RuntimeError(f"GitHub API error {response.status_code}")

        tree_data = response.json().get("tree", [])

        dir_count = sum(1 for item in tree_data if item["type"] == "tree")
        file_count = sum(1 for item in tree_data if item["type"] == "blob")

        return dir_count, file_count

    def directory_count(self, owner, repo):
        dirs, _ = self.get_directory_file_count(owner, repo)
        return dirs

    def file_count(self, owner, repo):
        _, files = self.get_directory_file_count(owner, repo)
        return files

    def stats(self, owner, repo):
        """
        Returns a consolidated view of key repository statistics.
        """

        data = self._repo_data(owner, repo)

        dirs, files = self.get_directory_file_count(owner, repo)

        return {
            "name": data.get("name"),
            "full_name": data.get("full_name"),
            "description": data.get("description"),
            "language": data.get("language"),
            "size_kb": data.get("size"),
            "visibility": data.get("visibility"),
            "default_branch": data.get("default_branch"),
            "stars": data.get("stargazers_count"),
            "forks": data.get("forks_count"),
            "watchers": data.get("watchers_count"),
            "directories": dirs,
            "files": files,
            "url": data.get("html_url"),
        }

    # -----------------------------
    # File classification
    # -----------------------------
    def _classify_files(self, owner, repo):

        repo_data = self._repo_data(owner, repo)
        branch = repo_data.get("default_branch", "main")

        url = f"https://api.github.com/repos/{owner}/{repo}/git/trees/{branch}?recursive=1"
        response = requests.get(url, headers=self.headers)

        if response.status_code != 200:
            raise RuntimeError(f"GitHub API error {response.status_code}: {response.text}")

        data = response.json()

        if data.get("truncated"):
            raise RuntimeError("Repo tree too large for single API call")

        tree = data.get("tree", [])

        classified_files = []
        unknown_extensions = set()  # move here

        for item in tree:

            if item["type"] == "blob":

                path = item["path"]
                file_name = os.path.basename(path)
                directory = os.path.dirname(path)

                ext = os.path.splitext(file_name)[1].lower()

                file_type = FILE_TYPES.get(ext, DEFAULT_FILE_TYPE)

                classified_files.append({
                    "File Type": file_type,
                    "Directory": directory,
                    "File Name": file_name
                })

                if ext not in FILE_TYPES:
                    unknown_extensions.add(ext)

        return classified_files, unknown_extensions

    # -----------------------------
    # Public CLI-exposed method
    # -----------------------------
    def file_types(self, owner, repo, output_csv="repo_files.csv"):

        files, unknown_exts = self._classify_files(owner, repo)

        if not files:
            return {"error": "No files found"}

        df = pd.DataFrame(files)

        # Prepare df for neo4j upload

        # --------------------------
        # Fixed output folder
        # --------------------------
        project_root = Path(__file__).resolve().parents[1]  # src/codebase_kg
        output_folder = project_root / "neo4j_imports"  # E:/Projects/.../neo4j_imports
        output_folder.mkdir(parents=True, exist_ok=True)

        # Final output path
        output_path = output_folder / output_csv

        # Save CSV
        df.to_csv(output_path, index=False, encoding="utf-8")

        # Frequency counts
        counts = df["File Type"].value_counts()

        total_files = len(df)

        # Directory count (unique directories)
        directory_count = df["Directory"].nunique()

        return {
            "total_files": total_files,
            "directory_count": directory_count,
            "output_csv": output_csv,
            "file_type_counts": counts.to_dict(),
            "unknown_extensions": sorted(list(unknown_exts))
        }

##### Extracting Zotero repo stats

###### Code: Choose repo to analyse

In [71]:
# Load Github PAT
env_file = Path.cwd().parent / ".env"
load_dotenv(env_file, override=True)

github_token = os.getenv("GITHUB_TOKEN")

# Create the service
github = GithubRepoService(token=github_token)

# Repository name
owner = "zotero"
repo = "zotero"

# Print result
result = github.stats(owner, repo)

# Pretty print result
for key, value in result.items():
    print(f"{key:20}: {value}")


name                : zotero
full_name           : zotero/zotero
description         : Zotero is a free, easy-to-use tool to help you collect, organize, annotate, cite, and share your research sources.
language            : JavaScript
size_kb             : 242352
visibility          : public
default_branch      : main
stars               : 15195
forks               : 1110
watchers            : 15195
directories         : 363
files               : 3489
url                 : https://github.com/zotero/zotero


##### Import what we know into Neo4j: KGs are built progressively
- directories, file names and file types form the base of the App KG